# Cortical Distributed Dipole Modeling via NIfTI-Based Masking

The number of dipoles is adjustable based on the spacing attribute. We then eliminate any dipole not associated with gray matter by applying a NIfTI mask. While the classic MNE-Python pipeline retains these dipoles, in our case, including non-gray matter areas interferes with our connectome construction.

## 🧪 - Experiment setup
- `ctx`: NIfTI cortical mask.
- `tkras2ras`: Spatial transform from FreeSurfer to Subject RAS.

In [3]:
# imports
import mindmap as mm
import mne

# readers
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("simf10")

# data
ctx = sr.load("ctx")
tkras2ras = sr.load("tkras2ras")

TypeError: SubjectReader._build_internal_path() missing 1 required positional argument: 'sub'

## 🧬 - Data Processing

Build Source Spaces

In [ ]:
# Setup source space
print("\033[95mSetup source space...\033[0m")
source_space = mne.setup_source_space(
    subject      = f"sub-{sr.sub}",
    spacing      = "oct6",                  # dipole spacing ('octN' is a mesh downsampling method)
    surface      = "white",                 # cortical mesh to use (white or pial)
    add_dist     = True,                    # geodesic distances between sources
    subjects_dir = dm.config["DIR_FS"],
    n_jobs       = 10,                      # number of threads
)

# mask with NIfTI
print("\033[95mMask medial wall...\033[0m")
source_space = mm.anatomy.source_space.filter_dipoles_by_nifti_mask(source_space, ctx, tkras2ras)

# summary
print("\n\033[92mSUMMARY\033[0m")
for src in source_space:
    print(f"Source space id: {src['id']}")
    print(f"Type: {src['type']}")
    print(f"Number of sources: {src['nuse']}\n")

SUMMARY
Source space id: 101 (FIFFV_MNE_SURF_LEFT_HEMI)
Type: surf
Number of sources: 3821

Source space id: 102 (FIFFV_MNE_SURF_RIGHT_HEMI)
Type: surf
Number of sources: 3826



## 📊 - Visualization

In [ ]:
mm.viz.plot_brain(
    source_space,
    overlay = ctx,
    tkras2ras = tkras2ras,
    mesh_params = {"show_src":True},
    overlay_params = {"color":"gray", "opacity":0.5},
).show()